In [124]:
# Import packages
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options

from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import ElementNotInteractableException

from bs4 import BeautifulSoup

import time, re, requests
import pandas as pd

In [125]:
# Open the browser without showing it on screen
opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("window-size=2560,1600")

# Open the Chrome browser
driver = webdriver.Chrome() #(options=opts)

# Navigate to the Project Gutenberg homepage
url = "https://www.gutenberg.org/"
driver.get(url)

# Set the implicit waiting time to 0.5 seconds
driver.implicitly_wait(0.5)
# Set the explicit wwaiting time to 20 seconds
wait = WebDriverWait(driver, 20)

In [126]:
# Find the navigation bar links
nav_bar = driver.find_elements(By.CSS_SELECTOR, ".main-links a")
if nav_bar:
    for link in nav_bar:
        # Print the navigation bar links
        print(link.text, link.get_attribute("href"))

 https://www.gutenberg.org/browse/scores/top
 https://www.gutenberg.org/ebooks/categories
Reading Lists https://www.gutenberg.org/ebooks/bookshelf/
Search Options https://www.gutenberg.org/ebooks/


In [127]:
# Navigate to the Reading Lists
reading_lists = driver.find_element(By.CSS_SELECTOR, "a[class='link-reading-lists']")
reading_lists.click()

# Pause execution for 5 seconds
time.sleep(5)

In [128]:
# Navigate to the Main Categories webpage
main_categories = driver.find_element(By.CSS_SELECTOR, "div[class='page_content'] p a")
main_categories.click()

# Pause execution for 5 seconds
time.sleep(5)

In [129]:
# Navigate to the Science-Fiction & Fantasy webpage
sf_fantasy = driver.find_element(By.XPATH, "/html/body/div[1]/div/div/div[1]/ul/li[14]/a")
sf_fantasy.click()

time.sleep(5)

In [138]:
def scrape_rows():
    """
    Scrape the rows from the table of a specific year.

    Returns:
        dict: Scraped table row values.
    """
    # Load Selenium's page_source into BeautifulSoup
    document = BeautifulSoup(driver.page_source, "html.parser")

    return {
        "titles": document.select(".title"),
        "authors": document.select(".subtitle"),
        "downloads": document.select(".extra"),
    }

In [139]:
def create_text():
    """
    Turn the scraped rows into text.

    Returns:
        dict: Table row values in text form.
    """
    # Assign scrape_rows() function to scrape_rows_result variable
    scrape_rows_result = scrape_rows()

    return {
        "title_text": [title.text for title in scrape_rows_result["titles"]],
        "author_text": [author.text for author in scrape_rows_result["authors"]],
        "download_text": [download.text for download in scrape_rows_result["downloads"]],
    }

In [142]:
def collect_data():
    """
    Iterates over table rows and scrapes it's content.
    
    Returns:
        List: Non-emtpy table rows.
    """
    # Assign create_text() function to create_text_result variable
    create_text_result = create_text()
    # Count the amount of table rows
    row_count = len(driver.find_elements(By.CSS_SELECTOR, ".booklink"))

    # Append text to list
    specific_page = []
    row_index = 0
    for row in range(row_count):
        specific_page.append({
            "Title": create_text_result["title_text"][row_index],
            "Author": create_text_result["author_text"][row_index],
            "Downloads": create_text_result["download_text"][row_index]
        })
        row_index += 1

    return specific_page

In [148]:
# Instantiate the df_novels DataFrame
df_novels = pd.DataFrame()

# Count the amount of books on the webpage
page_length = 2 # 169

# Iterate through the books on the webpage
for page in range(page_length):
    # Create the df_append DataFrame, based on the results of the scrape_page() function
    df_append = pd.DataFrame(collect_data())
    # Append the df_append DataFrame to the df_novels DataFrame
    df_novels = pd.concat([df_novels, df_append], ignore_index=True)

    next_page = driver.find_element(By.CSS_SELECTOR, "a[title='Go to the next page of results.']")
    if next_page:
        time.sleep(3)
        next_page.click()

# Show the df_novels DataFrame
df_novels

,Title,Author,Downloads
0,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,273991 downloads
1,The King in Yellow,Robert W. Chambers,70127 downloads
2,Dracula,Bram Stoker,42763 downloads
3,The Wonderful Wizard of Oz,L. Frank Baum,24500 downloads
4,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,21173 downloads
5,Gulliver's Travels into Several Remote Nations...,Jonathan Swift,17953 downloads
6,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,15125 downloads
7,Carmilla,Joseph Sheridan Le Fanu,13080 downloads
8,Rip Van Winkle,Washington Irving,11814 downloads
9,Twenty Thousand Leagues under the Sea,Jules Verne,10503 downloads
